In [96]:

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('perovskite_complete_2014_2026_v2.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()[:30]}...")
print(f"\nTotal columns: {len(df.columns)}")

# Check target PCE
if 'PCE' in df.columns:
    print(f"\nPCE stats:")
    print(df['PCE'].describe())
    print(f"PCE range: [{df['PCE'].min():.2f}, {df['PCE'].max():.2f}]")
    print(f"PCE >= 22% count: {(df['PCE'] >= 22).sum()}")
    print(f"PCE < 15% count: {(df['PCE'] < 15).sum()}")


Dataset shape: (7773, 511)

Columns: ['journal', 'pub_year', 'architecture', 'cell_area', 'composition_short', 'composition_long', 'band_gap', 'perovskite_thickness', 'dimension_3D', 'dimension_layers', 'dimension_2D', 'dimension_2D3D', 'depo_steps', 'main_solvent', 'sol_temp', 'sol_vol', 'thermal_temp', 'thermal_time', 'sub_temp', 'quenching', 'solvent_annealing', 'sol_anneal_temp', 'sol_anneal_time', 'etl_stack', 'etl_thickness', 'etl_solvents', 'etl_thermal_temp', 'etl_thermal_time', 'etl_sub_temp', 'etl_surf_treat']...

Total columns: 511

PCE stats:
count    7266.000000
mean       17.895997
std         5.726595
min         0.006000
25%        14.862500
50%        19.320000
75%        22.140000
max        27.490000
Name: PCE, dtype: float64
PCE range: [0.01, 27.49]
PCE >= 22% count: 1948
PCE < 15% count: 1846


In [97]:

# Identify additive-related columns
additive_cols = []
for col in df.columns:
    if any(kw in col.lower() for kw in ['solvent', 'sol_', 'ion_', 'coef', 'additive', 'dopant', 'passivation']):
        additive_cols.append(col)

print(f"Additive-related columns ({len(additive_cols)}):")
for c in additive_cols:
    print(f"  {c}")

# Also check for SMILES columns
smiles_cols = [c for c in df.columns if 'smiles' in c.lower()]
print(f"\nSMILES columns: {smiles_cols}")

# Check composition columns
comp_cols = [c for c in df.columns if 'composition' in c.lower() or 'ion' in c.lower()]
print(f"\nComposition/ion columns: {comp_cols}")


Additive-related columns (94):
  composition_short
  composition_long
  dimension_3D
  dimension_layers
  dimension_2D
  dimension_2D3D
  main_solvent
  sol_temp
  sol_vol
  solvent_annealing
  sol_anneal_temp
  sol_anneal_time
  etl_solvents
  htl_solvents
  htl_main_solvent
  composition_std
  A_ion_1
  A_coef_1
  A_ion_2
  A_coef_2
  A_ion_3
  A_coef_3
  A_ion_4
  A_coef_4
  A_ion_5
  A_coef_5
  B_ion_1
  B_coef_1
  B_ion_2
  B_coef_2
  B_ion_3
  B_coef_3
  C_ion_1
  C_coef_1
  C_ion_2
  C_coef_2
  C_ion_3
  C_coef_3
  C_ion_4
  C_coef_4
  C_ion_5
  C_coef_5
  A_ion_s_1
  A_ion_s_2
  A_ion_s_3
  A_ion_s_4
  A_ion_s_5
  A_coef_s_1
  A_coef_s_2
  A_coef_s_3
  A_coef_s_4
  A_coef_s_5
  B_ion_s_1
  B_ion_s_2
  B_ion_s_3
  B_coef_s_1
  B_coef_s_2
  B_coef_s_3
  C_ion_s_1
  C_ion_s_2
  C_ion_s_3
  C_ion_s_4
  C_ion_s_5
  C_coef_s_1
  C_coef_s_2
  C_coef_s_3
  C_coef_s_4
  C_coef_s_5
  sol_1
  sol_2
  sol_3
  sol_4
  sol_5
  sol_6
  sol_ratio_1
  sol_ratio_2
  sol_ratio_3
  sol_ratio_4
  s

In [98]:

# Step 1: Data preprocessing - select numeric features, handle missing values
# Remove non-numeric columns (metadata, text columns)
non_feature_cols = ['journal', 'pub_year', 'composition_short', 'composition_long', 'PCE']
# Also remove text-based columns
exclude_cols = non_feature_cols + [c for c in df.columns if df[c].dtype == 'object']

feature_cols = [c for c in df.columns if c not in exclude_cols]
print(f"Selected {len(feature_cols)} numeric feature columns")

# Check A_smiles and other text columns
smiles_related = [c for c in df.columns if 'smiles' in c.lower()]
print(f"SMILES columns to exclude: {smiles_related}")

# Exclude all object dtype columns
object_cols = [c for c in df.columns if df[c].dtype == 'object']
print(f"Object dtype columns ({len(object_cols)}): {object_cols[:20]}...")

# Final feature set
exclude_final = non_feature_cols + object_cols
feature_cols = [c for c in df.columns if c not in exclude_final]
print(f"\nFinal feature columns: {len(feature_cols)}")

# Extract feature matrix and target
X_raw = df[feature_cols].copy()
y = df['PCE'].copy()

print(f"\nFeature matrix shape: {X_raw.shape}")
print(f"Missing values per column (top 10):")
missing = X_raw.isnull().sum().sort_values(ascending=False)
print(missing.head(10))

# Fill missing values with median
X_filled = X_raw.fillna(X_raw.median())
print(f"\nAfter filling missing values: {X_filled.isnull().sum().sum()} remaining")


Selected 106 numeric feature columns
SMILES columns to exclude: ['A_smiles_1', 'A_smiles_2', 'A_smiles_3', 'A_smiles_4', 'A_smiles_5', 'B_smiles_1', 'B_smiles_2', 'B_smiles_3', 'C_smiles_1', 'C_smiles_2', 'C_smiles_3', 'C_smiles_4', 'C_smiles_5']
Object dtype columns (403): ['journal', 'architecture', 'composition_short', 'composition_long', 'dimension_3D', 'dimension_layers', 'dimension_2D', 'dimension_2D3D', 'main_solvent', 'sol_vol', 'quenching', 'solvent_annealing', 'etl_stack', 'etl_solvents', 'etl_surf_treat', 'htl_stack', 'htl_solvents', 'htl_main_solvent', 'htl_surf_treat', 'composition_std']...

Final feature columns: 106

Feature matrix shape: (7773, 106)
Missing values per column (top 10):
sol_anneal_time    7773
sol_anneal_temp    7773
C_ion_5            7773
C_coef_5           7773
C_src_cas_4        7773
C_cas_4            7773
C_source_4         7773
C_formula_4        7773
C_iupac_4          7773
A_cas_5            7773
dtype: int64

After filling missing values: 481926

In [99]:

# Better missing value handling - drop columns that are entirely NaN
X_clean = X_filled.dropna(axis=1, how='all')
print(f"After dropping all-NaN columns: {X_clean.shape}")

# Drop columns with >90% missing originally
missing_pct = X_raw.isnull().mean()
keep_cols = missing_pct[missing_pct < 0.9].index.tolist()
X_clean = X_filled[keep_cols]
print(f"After keeping columns with <90% missing: {X_clean.shape}")

# Final fill
X_clean = X_clean.fillna(X_clean.median())
# Any remaining NaN, fill with 0
X_clean = X_clean.fillna(0)

print(f"Final feature matrix: {X_clean.shape}")
print(f"Any NaN remaining: {X_clean.isnull().sum().sum()}")

# Check feature names
print(f"\nFeature columns ({len(X_clean.columns)}):")
for i, c in enumerate(X_clean.columns):
    print(f"  {i}: {c}")


After dropping all-NaN columns: (7773, 44)
After keeping columns with <90% missing: (7773, 27)
Final feature matrix: (7773, 27)
Any NaN remaining: 0

Feature columns (27):
  0: cell_area
  1: band_gap
  2: perovskite_thickness
  3: depo_steps
  4: thermal_temp
  5: thermal_time
  6: sub_temp
  7: etl_thickness
  8: etl_thermal_temp
  9: etl_thermal_time
  10: etl_sub_temp
  11: htl_thermal_temp
  12: backcontact_thick
  13: A_coef_1
  14: A_coef_2
  15: A_coef_3
  16: B_coef_1
  17: C_coef_1
  18: C_coef_2
  19: A_coef_s_1
  20: A_coef_s_2
  21: A_coef_s_3
  22: B_coef_s_1
  23: C_coef_s_1
  24: C_coef_s_2
  25: sub_temp_main
  26: sub_temp_part1


In [100]:

# We need to encode categorical features to get more dimensions
# Let's include categorical columns with reasonable cardinality
categorical_cols = []
for c in df.columns:
    if c in ['PCE', 'journal', 'pub_year']:
        continue
    if df[c].dtype == 'object':
        n_unique = df[c].nunique(dropna=True)
        if 1 < n_unique <= 50:  # Reasonable cardinality
            categorical_cols.append((c, n_unique))

print(f"Categorical columns with reasonable cardinality ({len(categorical_cols)}):")
for c, n in categorical_cols:
    print(f"  {c}: {n} unique values")


Categorical columns with reasonable cardinality (264):
  architecture: 8 unique values
  dimension_3D: 2 unique values
  dimension_layers: 7 unique values
  dimension_2D: 2 unique values
  dimension_2D3D: 2 unique values
  quenching: 2 unique values
  solvent_annealing: 2 unique values
  A_smiles_1: 27 unique values
  A_iupac_1: 27 unique values
  A_formula_1: 25 unique values
  A_source_1: 23 unique values
  A_cas_1: 8 unique values
  A_src_cas_1: 23 unique values
  A_ion_2: 40 unique values
  A_smiles_2: 22 unique values
  A_iupac_2: 22 unique values
  A_formula_2: 20 unique values
  A_source_2: 21 unique values
  A_cas_2: 9 unique values
  A_src_cas_2: 22 unique values
  A_ion_3: 14 unique values
  A_smiles_3: 10 unique values
  A_iupac_3: 10 unique values
  A_formula_3: 10 unique values
  A_source_3: 10 unique values
  A_cas_3: 3 unique values
  A_src_cas_3: 10 unique values
  A_ion_4: 7 unique values
  A_smiles_4: 5 unique values
  A_iupac_4: 5 unique values
  A_formula_4: 5 uniqu

In [101]:

# 与 1.ipynb（61e47ac1）一致：数值特征 + 低基数类别 one-hot（unique 2~15）
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
feature_cols_1 = [c for c in numeric_cols if c not in ['PCE', 'pub_year']]
cardinality = {c: df[c].nunique(dropna=True) for c in non_numeric_cols}
very_low_card = [c for c in non_numeric_cols if 2 <= cardinality[c] <= 15]

df_ohe = pd.get_dummies(df[very_low_card], dummy_na=False, drop_first=False)
X_base = df[feature_cols_1].fillna(0).values.astype(np.float64)
X_ohe = df_ohe.fillna(0).values.astype(np.float64)
X_tab = np.hstack([X_base, X_ohe])
_tab_names = feature_cols_1 + list(df_ohe.columns)
feature_df = pd.DataFrame(X_tab, columns=_tab_names, index=df.index)

print(
    f"与 1.ipynb 对齐: 数值维={len(feature_cols_1)}, "
    f"OHE 原始列={len(very_low_card)}, OHE 展开维={X_ohe.shape[1]}, 合计={feature_df.shape[1]}"
)
print(f"feature_df: {feature_df.shape}, NaN={feature_df.isnull().sum().sum()}")


与 1.ipynb 对齐: 数值维=106, OHE 原始列=208, OHE 展开维=1115, 合计=1221
feature_df: (7773, 1221), NaN=0


In [102]:

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

# Prepare data
X_full = feature_df.values
y_full = df['PCE'].values

# Remove samples with NaN PCE
valid_mask = ~np.isnan(y_full)
X_full = X_full[valid_mask]
y_full = y_full[valid_mask]

print(f"Valid samples: {len(y_full)}")
print(f"PCE range: [{y_full.min():.2f}, {y_full.max():.2f}]")

# Standardize features
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X_full)

# PCA：累计解释方差 >= 95%（主成分个数由数据自动确定）
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)
n_pca = X_pca.shape[1]
print(f"\nPCA n_components={n_pca}, cumulative explained variance: {pca.explained_variance_ratio_.sum():.4f}")
print(f"PCA shape: {X_pca.shape}")

# Flow Matching：在列标准化后的潜空间训练，使 x_0~N(0,I) 与线性插值目标一致
latent_scaler = StandardScaler()
X_pca_z = latent_scaler.fit_transform(X_pca)
print("Latent z-score: per-PC mean≈0, std≈1 (用于 CFM)")

# Normalize PCE to [0, 1] as condition
y_min, y_max = y_full.min(), y_full.max()
y_norm = (y_full - y_min) / (y_max - y_min)
print(f"PCE normalized range: [{y_norm.min():.4f}, {y_norm.max():.4f}]")

# Train-test split（索引划分，便于与 LGBM teacher 对齐）
_idx = np.arange(len(X_pca))
split_tr_idx, split_te_idx = train_test_split(_idx, test_size=0.2, random_state=42)
X_train, X_test = X_pca[split_tr_idx], X_pca[split_te_idx]
X_train_z, X_test_z = X_pca_z[split_tr_idx], X_pca_z[split_te_idx]
y_train, y_test = y_norm[split_tr_idx], y_norm[split_te_idx]
y_train_raw, y_test_raw = y_full[split_tr_idx], y_full[split_te_idx]
print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")


Valid samples: 7266
PCE range: [0.01, 27.49]

PCA explained variance ratio: 0.4777
PCA shape: (7266, 50)
Latent z-score: per-PC mean≈0, std≈1 (用于 CFM)
PCE normalized range: [0.0000, 1.0000]

Train: (5812, 50), Test: (1454, 50)


In [103]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Set random seed
torch.manual_seed(42)
np.random.seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Convert to tensors（X_train_t / X_test_t = 原始 PCA，供 PCE 头；*_fm_t = z 空间，供 CFM）
X_train_t = torch.FloatTensor(X_train).to(device)
X_train_fm_t = torch.FloatTensor(X_train_z).to(device)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1).to(device)
X_test_t = torch.FloatTensor(X_test).to(device)
X_test_fm_t = torch.FloatTensor(X_test_z).to(device)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1).to(device)

print(f"X_train (PCA): {X_train_t.shape}, X_train_fm (z): {X_train_fm_t.shape}, y_train: {y_train_t.shape}")

# Velocity Field Network
class VelocityField(nn.Module):
    def __init__(self, x_dim=50, c_dim=1, hidden_dim=256):
        super().__init__()
        self.x_dim = x_dim
        self.c_dim = c_dim
        
        # Time embedding
        self.time_embed = nn.Sequential(
            nn.Linear(1, 64),
            nn.SiLU(),
            nn.Linear(64, 64)
        )
        
        # Main network: input = x (n_pca) + c (1) + t_embed (64)
        self.net = nn.Sequential(
            nn.Linear(x_dim + c_dim + 64, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 128),
            nn.SiLU(),
            nn.Linear(128, x_dim)
        )
    
    def forward(self, x, c, t):
        # x: (B, x_dim), c: (B, c_dim), t: (B, 1)
        t_embed = self.time_embed(t)
        inp = torch.cat([x, c, t_embed], dim=-1)
        return self.net(inp)

# Initialize model
model = VelocityField(x_dim=n_pca, c_dim=1, hidden_dim=256).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")

# Optimizer（调度周期与训练轮数一致，避免 LR 病态振荡）
optimizer = optim.Adam(model.parameters(), lr=1e-3)
FM_EPOCHS = 600
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FM_EPOCHS, eta_min=1e-5)

print("Model initialized successfully")


Device: cuda
X_train (PCA): torch.Size([5812, 50]), X_train_fm (z): torch.Size([5812, 50]), y_train: torch.Size([5812, 1])
Model parameters: 139122
Model initialized successfully


In [104]:

# Conditional Flow Matching Training
def train_cfm_epoch(model, X_data, y_data, optimizer, batch_size=512):
    model.train()
    dataset = TensorDataset(X_data, y_data)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    total_loss = 0
    n_batches = 0
    
    for x_batch, y_batch in loader:
        optimizer.zero_grad()
        
        batch_size_actual = x_batch.shape[0]
        
        # Sample random time t ~ U[0,1]
        t = torch.rand(batch_size_actual, 1).to(device)
        
        # Sample noise from prior: x_0 ~ N(0, I)
        x_0 = torch.randn_like(x_batch)
        
        # Conditional probability path: x_t = t * x_1 + (1-t) * x_0
        x_t = t * x_batch + (1 - t) * x_0
        
        # Target velocity: u_t = x_1 - x_0 (for linear path)
        target_vel = x_batch - x_0
        
        # Predict velocity
        pred_vel = model(x_t, y_batch, t)
        
        # MSE loss
        loss = ((pred_vel - target_vel) ** 2).mean()
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    return total_loss / n_batches

# Training loop（速度场在 z 空间训练；保留 epoch 最优权重）
print("Training Conditional Flow Matching model...")
try:
    n_epochs = FM_EPOCHS
except NameError:
    n_epochs = 600
best_loss = float('inf')
best_state = None
train_losses = []

for epoch in range(n_epochs):
    loss = train_cfm_epoch(model, X_train_fm_t, y_train_t, optimizer, batch_size=512)
    train_losses.append(loss)
    scheduler.step()
    if loss < best_loss:
        best_loss = loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss:.6f}, Best: {best_loss:.6f}")

if best_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f"已加载最优 checkpoint（train loss={best_loss:.6f}），非最后一轮权重。")
print(f"\nTraining complete. Best loss: {best_loss:.6f}")

@torch.no_grad()
def sample_conditional_cfm(velocity_model, c_norm, batch_size, n_steps=100):
    """条件 CFM：从 x_0~N(0,I) 欧拉积分到 t=1，返回 z 空间样本 (numpy)。"""
    velocity_model.eval()
    x = torch.randn(batch_size, velocity_model.x_dim, device=device)
    c = torch.full((batch_size, 1), float(c_norm), device=device)
    dt = 1.0 / n_steps
    for k in range(n_steps):
        t = torch.full((batch_size, 1), k * dt, device=device)
        v = velocity_model(x, c, t)
        x = x + dt * v
    return x.cpu().numpy()



Training Conditional Flow Matching model...
Epoch 5/600, Loss: 1.960085, Best: 1.960085
Epoch 10/600, Loss: 1.606834, Best: 1.606834
Epoch 15/600, Loss: 1.306782, Best: 1.306782
Epoch 20/600, Loss: 1.145239, Best: 1.145239
Epoch 25/600, Loss: 1.074365, Best: 1.066703
Epoch 30/600, Loss: 1.022546, Best: 1.015262
Epoch 35/600, Loss: 0.953325, Best: 0.953325
Epoch 40/600, Loss: 0.960016, Best: 0.890967
Epoch 45/600, Loss: 0.900915, Best: 0.884305
Epoch 50/600, Loss: 0.866580, Best: 0.840597
Epoch 55/600, Loss: 0.884726, Best: 0.810021
Epoch 60/600, Loss: 0.811363, Best: 0.810021
Epoch 65/600, Loss: 0.832018, Best: 0.788484
Epoch 70/600, Loss: 0.813399, Best: 0.779166
Epoch 75/600, Loss: 0.737335, Best: 0.731296
Epoch 80/600, Loss: 0.706629, Best: 0.706629
Epoch 85/600, Loss: 0.741437, Best: 0.706629
Epoch 90/600, Loss: 0.718010, Best: 0.706629
Epoch 95/600, Loss: 0.742325, Best: 0.706629
Epoch 100/600, Loss: 0.683819, Best: 0.683819
Epoch 105/600, Loss: 0.721588, Best: 0.679443
Epoch 110/

In [105]:
# 不再落盘 CFM 权重；需要时请自行 torch.save（避免产生 fm 相关结果文件）
print("已跳过 cfm_model_final.pth 保存。")


已跳过 cfm_model_final.pth 保存。


In [106]:

import os
import joblib
import torch.nn.functional as F
from sklearn.metrics import r2_score, mean_squared_error

os.makedirs('output', exist_ok=True)

# ---------- PCE：490d 教师（run_full_pipeline 的 smart 多模型加权，或旧 LGBM+ET / 单 LGBM） ----------
_bpath = 'output/pce_lgbm_490d_bundle.joblib'
_zpath = 'output/yearly_sequence_data.npz'
if not os.path.isfile(_bpath):
    raise FileNotFoundError('请先运行 1.ipynb 至保存 yearly_sequence_data.npz 与 pce_lgbm_490d_bundle.joblib')
if not os.path.isfile(_zpath):
    raise FileNotFoundError('缺少 output/yearly_sequence_data.npz，请先运行 1.ipynb')

bundle = joblib.load(_bpath)
seq = np.load(_zpath)
if 'sorted_source_index' not in seq.files:
    raise ValueError('yearly_sequence_data.npz 需含 sorted_source_index，请重新运行 1.ipynb 保存单元格')

sorted_source_index = seq['sorted_source_index']
X490_hw = np.hstack([seq['X_numeric'], seq['lstm_features'], seq['transformer_features'], seq['gru_features']])
scaler_490 = bundle['scaler_490']

valid_positions = np.where(valid_mask)[0]
orig_ids = df.index[valid_positions].astype(np.int64).values
_inv = {int(sorted_source_index[j]): j for j in range(len(sorted_source_index))}
X490_rows = np.stack([X490_hw[_inv[int(oid)]] for oid in orig_ids])
_X490s = scaler_490.transform(X490_rows)
if bundle.get('teacher_models') and bundle.get('teacher_weights'):
    y_lgb_full = np.zeros(len(X490_rows), dtype=np.float64)
    for _nm, _w in bundle['teacher_weights'].items():
        if _w <= 0:
            continue
        y_lgb_full += _w * bundle['teacher_models'][_nm].predict(_X490s)
else:
    _lgb = bundle.get('lgbm')
    if _lgb is None:
        raise ValueError('bundle 中缺少教师模型，请运行 run_full_pipeline 生成 pce_lgbm_490d_bundle.joblib')
    _et = bundle.get('extratrees')
    _wl = float(bundle.get('teacher_w_lgbm', 1.0))
    _we = float(bundle.get('teacher_w_et', 0.0))
    if _et is not None and _wl + _we > 0:
        _s = _wl + _we
        y_lgb_full = (_wl / _s) * _lgb.predict(_X490s) + (_we / _s) * _et.predict(_X490s)
    else:
        y_lgb_full = _lgb.predict(_X490s)

class PCAPCEDistiller(nn.Module):
    def __init__(self, n_in, hidden=(256, 128), dropout=0.12):
        super().__init__()
        h1, h2 = hidden
        self.net = nn.Sequential(
            nn.Linear(n_in, h1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h2, 1),
        )

    def forward(self, x):
        return self.net(x)

_n_pca = X_pca.shape[1]
pce_predictor = PCAPCEDistiller(_n_pca).to(device)
_opt = torch.optim.AdamW(pce_predictor.parameters(), lr=8e-4, weight_decay=1e-4)

X_tr_t = torch.FloatTensor(X_pca[split_tr_idx]).to(device)
y_tr_teacher = torch.FloatTensor(y_lgb_full[split_tr_idx]).reshape(-1, 1).to(device)
X_va_t = torch.FloatTensor(X_pca[split_te_idx]).to(device)
y_va_teacher = torch.FloatTensor(y_lgb_full[split_te_idx]).reshape(-1, 1).to(device)

best_state, best_val = None, float('inf')
for _ep in range(600):
    pce_predictor.train()
    _opt.zero_grad()
    _loss = F.mse_loss(pce_predictor(X_tr_t), y_tr_teacher)
    _loss.backward()
    torch.nn.utils.clip_grad_norm_(pce_predictor.parameters(), 1.0)
    _opt.step()
    pce_predictor.eval()
    with torch.no_grad():
        _v = F.mse_loss(pce_predictor(X_va_t), y_va_teacher).item()
    if _v < best_val:
        best_val = _v
        best_state = {k: v.detach().cpu().clone() for k, v in pce_predictor.state_dict().items()}

if best_state is not None:
    pce_predictor.load_state_dict({k: v.to(device) for k, v in best_state.items()})
pce_predictor.eval()

with torch.no_grad():
    pred_te = pce_predictor(X_va_t).cpu().numpy().flatten()
r2 = r2_score(y_full[split_te_idx], pred_te)
rmse = float(np.sqrt(mean_squared_error(y_full[split_te_idx], pred_te)))
pred_lgb_te = lgbm.predict(scaler_490.transform(X490_rows[split_te_idx]))
r2_lgb = r2_score(y_full[split_te_idx], pred_lgb_te)
print(f"LightGBM(490d，1.ipynb) 同划分测试 R²={r2_lgb:.4f}（教师 vs 真实 PCE）")
print(f"MLP 蒸馏：验证集上对教师 MSE={best_val:.5f}（越小越贴近 LGBM）")
print(f"MLP(PCA→PCE) 对真实 PCE 测试 R²={r2:.4f}, RMSE={rmse:.4f}")


LightGBM(490d，1.ipynb) 同划分测试 R²=0.9797（教师 vs 真实 PCE）
MLP 蒸馏：验证集上对教师 MSE=14.88887（越小越贴近 LGBM）
MLP(PCA→PCE) 对真实 PCE 测试 R²=0.5236, RMSE=3.9456


In [107]:

# Part 2: Additive Perturbation Analysis
# Identify additive-related dimensions in original feature space
additive_feature_indices = []
additive_feature_names = []

for i, col in enumerate(feature_df.columns):
    if any(kw in col.lower() for kw in ['ion', 'sol', 'solvent', 'add']):
        additive_feature_indices.append(i)
        additive_feature_names.append(col)

print(f"Additive-related features ({len(additive_feature_indices)}):")
for idx, name in zip(additive_feature_indices, additive_feature_names):
    print(f"  Index {idx}: {name}")

# Step 2: Compute additive direction gradients
# For each additive dimension, perturb in positive and negative direction
# and measure PCE change

pce_predictor.eval()

def compute_pce_from_features(features_orig):
    """Convert original features to PCA and predict PCE"""
    features_scaled = scaler_X.transform(features_orig)
    features_pca = pca.transform(features_scaled)
    features_t = torch.FloatTensor(features_pca).to(device)
    with torch.no_grad():
        pce = pce_predictor(features_t).cpu().numpy().flatten()
    return pce

# Use a subset of data for gradient computation (full data = 7266 samples)
X_full_np = feature_df.values
n_samples = min(2000, len(X_full_np))
np.random.seed(42)
sample_indices = np.random.choice(len(X_full_np), n_samples, replace=False)
X_sample = X_full_np[sample_indices]

print(f"\nComputing gradients on {n_samples} samples...")

# For each additive dimension, compute average gradient
delta = 0.1
gradients = {}

for idx, name in zip(additive_feature_indices, additive_feature_names):
    grads = []
    
    for j in range(len(X_sample)):
        x_j = X_sample[j].copy()
        
        # Get base PCE
        base_pce = compute_pce_from_features(x_j.reshape(1, -1))[0]
        
        # Positive perturbation
        x_pos = x_j.copy()
        std_val = np.std(X_full_np[:, idx])
        if std_val > 0:
            x_pos[idx] += delta * std_val
            pce_pos = compute_pce_from_features(x_pos.reshape(1, -1))[0]
            
            # Negative perturbation
            x_neg = x_j.copy()
            x_neg[idx] -= delta * std_val
            pce_neg = compute_pce_from_features(x_neg.reshape(1, -1))[0]
            
            # Gradient
            grad = (pce_pos - pce_neg) / (2 * delta)
            grads.append(grad)
    
    if grads:
        gradients[name] = {
            'mean_grad': np.mean(grads),
            'std_grad': np.std(grads),
            'abs_mean_grad': np.abs(np.mean(grads)),
            'index': idx
        }

print(f"Computed gradients for {len(gradients)} additive dimensions")


Additive-related features (254):
  Index 4: sol_temp
  Index 8: sol_anneal_temp
  Index 9: sol_anneal_time
  Index 32: C_ion_4
  Index 40: C_ion_5
  Index 56: C_ion_s_4
  Index 57: C_ion_s_5
  Index 63: sol_5
  Index 64: sol_6
  Index 65: sol_ratio_4
  Index 66: sol_ratio_5
  Index 67: sol_ratio_6
  Index 68: sol_add_6
  Index 76: add_raw_4
  Index 77: add_raw_5
  Index 78: add_smi_4
  Index 79: add_smi_5
  Index 91: add_raw_4_main
  Index 92: add_raw_5_main
  Index 93: add_smi_4_main
  Index 94: add_smi_5_main
  Index 114: dimension_3D_False
  Index 115: dimension_3D_True
  Index 116: dimension_layers_0D
  Index 117: dimension_layers_1D
  Index 118: dimension_layers_2
  Index 119: dimension_layers_2D
  Index 120: dimension_layers_2D/3D
  Index 121: dimension_layers_3
  Index 122: dimension_layers_3D
  Index 123: dimension_2D_False
  Index 124: dimension_2D_True
  Index 125: dimension_2D3D_False
  Index 126: dimension_2D3D_True
  Index 129: solvent_annealing_False
  Index 130: solvent_

In [108]:

# More efficient gradient computation - vectorize where possible
pce_predictor.eval()

# Get scaler std for each additive dimension
scaler_std = np.sqrt(scaler_X.var_)

# Work in PCA space directly for efficiency
X_full_scaled = scaler_X.transform(feature_df.values)
X_full_pca = pca.transform(X_full_scaled)

# Convert predictor to eval and prepare
n_samples_grad = 500  # Use subset for speed
np.random.seed(42)
sample_idx = np.random.choice(len(X_full_pca), n_samples_grad, replace=False)
X_grad_pca = X_full_pca[sample_idx]

# Predict base PCE for all samples at once
X_grad_tensor = torch.FloatTensor(X_grad_pca).to(device)
with torch.no_grad():
    base_pces = pce_predictor(X_grad_tensor).cpu().numpy().flatten()

print(f"Base PCE range: [{base_pces.min():.2f}, {base_pces.max():.2f}]")
print(f"Computing gradients for {len(additive_feature_indices)} dimensions x {n_samples_grad} samples...")

gradients = {}
delta = 0.1

for feat_idx, name in zip(additive_feature_indices, additive_feature_names):
    grads = []
    std_val = scaler_std[feat_idx]
    
    if std_val <= 0:
        continue
    
    # Get the PCA loading for this feature dimension
    # The feature perturbation in original space maps to PCA space
    # delta_orig * loading_vector = delta_pca
    loading = pca.components_[:, feat_idx]  # (50,) - how this feature contributes to each PC
    
    for j in range(n_samples_grad):
        # Positive perturbation in PCA space
        delta_pca = delta * std_val * loading
        x_pos = X_grad_pca[j] + delta_pca
        x_neg = X_grad_pca[j] - delta_pca
        
        # Predict PCE
        x_pos_t = torch.FloatTensor(x_pos).unsqueeze(0).to(device)
        x_neg_t = torch.FloatTensor(x_neg).unsqueeze(0).to(device)
        
        with torch.no_grad():
            pce_pos = pce_predictor(x_pos_t).item()
            pce_neg = pce_predictor(x_neg_t).item()
        
        grad = (pce_pos - pce_neg) / (2 * delta)
        grads.append(grad)
    
    if grads:
        gradients[name] = {
            'mean_grad': np.mean(grads),
            'std_grad': np.std(grads),
            'abs_mean_grad': np.abs(np.mean(grads)),
            'index': feat_idx
        }

# Sort by mean gradient
sorted_grads = sorted(gradients.items(), key=lambda x: x[1]['mean_grad'], reverse=True)

print(f"\n--- Top 10 Most Beneficial Additive Directions ---")
for name, info in sorted_grads[:10]:
    print(f"  {name}: mean_grad={info['mean_grad']:.4f} ± {info['std_grad']:.4f}")

print(f"\n--- Top 10 Most Harmful Additive Directions ---")
for name, info in sorted_grads[-10:]:
    print(f"  {name}: mean_grad={info['mean_grad']:.4f} ± {info['std_grad']:.4f}")


Base PCE range: [0.38, 25.90]
Computing gradients for 254 dimensions x 500 samples...

--- Top 10 Most Beneficial Additive Directions ---
  dimension_layers_3D: mean_grad=0.0840 ± 0.0433
  B_ion_2_Sn: mean_grad=0.0831 ± 0.0483
  B_ion_s_2_Sn: mean_grad=0.0831 ± 0.0483
  C_ion_1_I: mean_grad=0.0486 ± 0.0761
  C_ion_s_1_I: mean_grad=0.0486 ± 0.0761
  A_ion_s_3_MA: mean_grad=0.0459 ± 0.0371
  A_ion_3_MA: mean_grad=0.0458 ± 0.0369
  add_raw_1_type_unknown: mean_grad=0.0368 ± 0.0211
  A_ion_3_PEA: mean_grad=0.0351 ± 0.0189
  A_ion_s_3_PEA: mean_grad=0.0351 ± 0.0189

--- Top 10 Most Harmful Additive Directions ---
  sub_clean_step1_Mucasol: mean_grad=-0.0675 ± 0.0309
  sub_clean_step1_main_Mucasol: mean_grad=-0.0675 ± 0.0309
  sub_clean_step1_part1_Mucasol: mean_grad=-0.0675 ± 0.0309
  add_raw_1_type_pure: mean_grad=-0.0786 ± 0.0356
  C_ion_1_Cl: mean_grad=-0.1010 ± 0.0538
  C_ion_s_1_Cl: mean_grad=-0.1010 ± 0.0538
  dimension_2D3D_False: mean_grad=-0.1042 ± 0.0483
  solvent_annealing_False:

In [109]:

# The PCE predictor is extrapolating beyond reasonable range
# Let's recompute with clipping and smaller step size

valid_mask_full = ~np.isnan(df['PCE'].values)
y_full_valid = df.loc[valid_mask_full, 'PCE'].values.astype(float)
X_pca_valid = X_full_pca[valid_mask_full]

np.random.seed(42)
path_indices = np.random.permutation(len(y_full_valid))

top_5_directions = sorted_grads[:5]

path_results = []
max_steps = 8
step_size = 0.02  # Smaller step

for i, idx in enumerate(path_indices[:100]):
    x_current = X_pca_valid[idx].copy()
    y_current = y_full_valid[idx]
    trajectory = [y_current]
    
    for step in range(max_steps):
        best_grad = -float('inf')
        best_delta_pca = None
        y_new = y_current
        
        for feat_name, info in top_5_directions:
            feat_idx = info['index']
            std_val = scaler_std[feat_idx]
            loading = pca.components_[:, feat_idx]
            
            delta_pca = step_size * std_val * loading
            x_test = x_current + delta_pca
            
            x_test_t = torch.FloatTensor(x_test).unsqueeze(0).to(device)
            with torch.no_grad():
                y_test = pce_predictor(x_test_t).item()
            y_test = np.clip(y_test, 0, 30)  # Clip to reasonable range
            
            grad = y_test - y_current
            if grad > best_grad:
                best_grad = grad
                best_delta_pca = delta_pca
                y_new = y_test
        
        if best_grad <= 0.05:  # Minimum improvement threshold
            break
        
        x_current += best_delta_pca
        y_current = y_new
        trajectory.append(y_current)
    
    path_results.append({
        'sample_id': idx,
        'initial_pce': y_full_valid[idx],
        'final_pce': y_current,
        'n_steps': len(trajectory) - 1,
        'pce_improvement': y_current - y_full_valid[idx],
        'trajectory': trajectory
    })

avg_improvement = np.mean([r['pce_improvement'] for r in path_results])
avg_steps = np.mean([r['n_steps'] for r in path_results])
n_reached_high = sum(1 for r in path_results if r['final_pce'] >= 20)

print(f"=== Optimal Transport Path Summary (Revised) ===")
print(f"Average PCE improvement: {avg_improvement:.2f}%")
print(f"Average steps needed: {avg_steps:.1f}")
print(f"Samples reaching PCE >= 20%: {n_reached_high}/{len(path_results)}")
print(f"Initial PCE: mean={np.mean([r['initial_pce'] for r in path_results]):.2f}%")
print(f"Final PCE: mean={np.mean([r['final_pce'] for r in path_results]):.2f}%")
print(f"Final PCE: median={np.median([r['final_pce'] for r in path_results]):.2f}%")


=== Optimal Transport Path Summary (Revised) ===
Average PCE improvement: 1.28%
Average steps needed: 0.4
Samples reaching PCE >= 20%: 60/100
Initial PCE: mean=18.55%
Final PCE: mean=19.83%
Final PCE: median=20.76%


In [110]:

# Fix: X_full_pca has 7773 rows (from feature_df), but y_full_valid has 7266 rows
# Let's align them
print(f"X_full_pca shape: {X_full_pca.shape}")
print(f"y_full shape: {y_full.shape}")
print(f"feature_df shape: {feature_df.shape}")

# Recompute with aligned indices
y_aligned = df['PCE'].values
valid_mask_full = ~np.isnan(y_aligned)
X_aligned = X_full_pca[valid_mask_full]  # This should match y_full
print(f"X_aligned shape: {X_aligned.shape}")
print(f"y_full shape: {y_full.shape}")

# Now use X_aligned for high PCE analysis
high_pce_mask = y_full >= 22
X_high_pce = X_aligned[high_pce_mask]
print(f"High PCE samples (>=22%): {len(X_high_pce)}")

# Compute centroid
high_pce_centroid = X_high_pce.mean(axis=0)
print(f"High PCE centroid shape: {high_pce_centroid.shape}")

# Predict PCE at centroid
centroid_t = torch.FloatTensor(high_pce_centroid).unsqueeze(0).to(device)
with torch.no_grad():
    centroid_pce = pce_predictor(centroid_t).item()
print(f"PCE at centroid: {centroid_pce:.2f}%")
empirical_high_mean = float(y_full[high_pce_mask].mean())
print(f"簇内真实 PCE 均值 (>=22%): {empirical_high_mean:.2f}%")
print("(潜空间质心的回归预测不必等于高 PCE 簇均值：非线性与流形弯曲。)\n")

# Map centroid back to original feature space
centroid_orig = pca.inverse_transform(high_pce_centroid.reshape(1, -1))
centroid_orig_unscaled = scaler_X.inverse_transform(centroid_orig)[0].copy()
feature_names = feature_df.columns.tolist()
for col_name, lo, hi, use_abs in [
    ('cell_area', 0.04, 2.0, True),
    ('perovskite_thickness', 100, 1000, True),
    ('thermal_temp', 50, 300, False),
]:
    if col_name in feature_names:
        i = feature_names.index(col_name)
        v = centroid_orig_unscaled[i]
        centroid_orig_unscaled[i] = np.clip(np.abs(v), lo, hi) if use_abs else np.clip(v, lo, hi)

# Compare to global mean
global_mean = feature_df[valid_mask_full].mean().values

diff_features = []
for i, name in enumerate(feature_names):
    diff = centroid_orig_unscaled[i] - global_mean[i]
    diff_features.append((name, diff, centroid_orig_unscaled[i], global_mean[i]))

diff_features.sort(key=lambda x: abs(x[1]), reverse=True)
print(f"\nTop 15 different features at high PCE centroid:")
for name, diff, cent, glob in diff_features[:15]:
    print(f"  {name}: centroid={cent:.2f}, global={glob:.2f}, diff={diff:.2f}")


X_full_pca shape: (7773, 50)
y_full shape: (7266,)
feature_df shape: (7773, 1221)
X_aligned shape: (7266, 50)
y_full shape: (7266,)
High PCE samples (>=22%): 1948
High PCE centroid shape: (50,)
PCE at centroid: 13.84%
簇内真实 PCE 均值 (>=22%): 23.74%
(潜空间质心的回归预测不必等于高 PCE 簇均值：非线性与流形弯曲。)


Top 15 different features at high PCE centroid:
  sub_thickness: centroid=509.25, global=422.00, diff=87.25
  sub_thickness_main: centroid=509.25, global=422.00, diff=87.25
  sub_thickness_part1: centroid=509.25, global=422.00, diff=87.25
  backcontact_thick: centroid=445.78, global=404.50, diff=41.27
  thermal_time: centroid=436.28, global=405.61, diff=30.67
  etl_thermal_time: centroid=322.54, global=298.86, diff=23.68
  thermal_temp: centroid=50.00, global=41.93, diff=8.07
  perovskite_thickness: centroid=136.29, global=143.11, diff=-6.83
  htl_thermal_time: centroid=95.95, global=90.95, diff=5.00
  htl_sub_temp: centroid=12.36, global=16.20, diff=-3.83
  htl_thermal_temp: centroid=12.42, global=15.09, d

In [111]:

# Step 3: 条件 Flow Matching 采样（与训练速度场一致：z 空间 ODE → 逆 z → PCA 原空间）
n_virtual = 100
np.random.seed(42)
print(f"--- Generating {n_virtual} virtual recipes (CFM Euler, not centroid+noise) ---")

_mask_high = (y_full >= 22)
target_c = float(y_norm[_mask_high].mean())
print(f"CFM 条件 c = 高 PCE(>=22) 子集上归一化 PCE 均值: {target_c:.4f}")

virtual_z = sample_conditional_cfm(model, target_c, n_virtual, n_steps=100)
virtual_pca = latent_scaler.inverse_transform(virtual_z)
virtual_pca = virtual_pca + np.random.normal(0, 0.05, size=virtual_pca.shape)

# Predict PCE for virtual recipes
virtual_tensor = torch.FloatTensor(virtual_pca).to(device)
with torch.no_grad():
    virtual_pces = pce_predictor(virtual_tensor).cpu().numpy().flatten()

virtual_pces = np.clip(virtual_pces, 0, 30)

print(f"Virtual PCE predictions:")
print(f"  Range: [{virtual_pces.min():.2f}, {virtual_pces.max():.2f}]")
print(f"  Mean: {virtual_pces.mean():.2f}, Median: {np.median(virtual_pces):.2f}")
print(f"  PCE >= 20%: {(virtual_pces >= 20).sum()}/{n_virtual}")
print(f"  PCE >= 22%: {(virtual_pces >= 22).sum()}/{n_virtual}")

# Map back to original feature space
virtual_orig = pca.inverse_transform(virtual_pca)
virtual_orig_unscaled = scaler_X.inverse_transform(virtual_orig)

# Create DataFrame for virtual recipes
virtual_recipe_df = pd.DataFrame(virtual_orig_unscaled, columns=feature_names)
# PCA 逆变换后施加物理约束，减轻伪负面积等伪影（与后续 NN 距离所用矩阵对齐）
for col_name, lo, hi, use_abs in [
    ('cell_area', 0.04, 2.0, True),
    ('perovskite_thickness', 100, 1000, True),
    ('thermal_temp', 50, 300, False),
]:
    if col_name in virtual_recipe_df.columns:
        v = virtual_recipe_df[col_name].astype(float).values
        virtual_recipe_df[col_name] = np.clip(np.abs(v), lo, hi) if use_abs else np.clip(v, lo, hi)
virtual_orig_unscaled = virtual_recipe_df[feature_names].values
virtual_recipe_df['PCE_predicted'] = virtual_pces

# Save virtual recipes
virtual_recipe_df.to_csv('output/virtual_recipes.csv', index=False)
print(f"\nSaved: output/virtual_recipes.csv")

# Also save aligned latent space
df_valid = df[valid_mask_full].copy()
df_valid.to_csv('output/df_valid_aligned.csv', index=False)
print(f"Saved aligned valid data: {df_valid.shape}")


--- Generating 100 virtual recipes (CFM Euler, not centroid+noise) ---
CFM 条件 c = 高 PCE(>=22) 子集上归一化 PCE 均值: 0.8637
Virtual PCE predictions:
  Range: [8.88, 27.15]
  Mean: 18.72, Median: 18.70
  PCE >= 20%: 37/100
  PCE >= 22%: 25/100

Saved: output/virtual_recipes.csv
Saved aligned valid data: (7266, 511)


In [112]:

# Part 4: Functional Group Commonality Analysis
# Step 1: Extract functional groups from real high PCE samples

# Check available SMILES columns
smiles_cols = ['A_smiles_1', 'A_smiles_2', 'A_smiles_3', 'A_smiles_4', 'A_smiles_5',
               'B_smiles_1', 'B_smiles_2', 'B_smiles_3',
               'C_smiles_1', 'C_smiles_2', 'C_smiles_3', 'C_smiles_4', 'C_smiles_5']
available_smiles = [c for c in smiles_cols if c in df.columns]
print(f"Available SMILES columns: {available_smiles}")

# Get real high PCE samples
df_valid_reset = df[valid_mask_full].reset_index(drop=True)
high_pce_real = df_valid_reset[df_valid_reset['PCE'] >= 22].copy()
print(f"Real high PCE samples (>=22%): {len(high_pce_real)}")

from rdkit import Chem
print("RDKit: SMARTS 子结构匹配（须安装 rdkit）")


Available SMILES columns: ['A_smiles_1', 'A_smiles_2', 'A_smiles_3', 'A_smiles_4', 'A_smiles_5', 'B_smiles_1', 'B_smiles_2', 'B_smiles_3', 'C_smiles_1', 'C_smiles_2', 'C_smiles_3', 'C_smiles_4', 'C_smiles_5']
Real high PCE samples (>=22%): 1948
RDKit: SMARTS 子结构匹配（须安装 rdkit）


In [113]:


# RDKit SMARTS 官能团检测（51 类）；下游仍使用 FG_PATTERNS.keys()

FG_SMARTS = {
    'amine_primary': ['[NX3;H2;!$(NC=O)]'],
    'amine_secondary': ['[NX3;H1;!$(NC=O)]'],
    'amine_tertiary': ['[NX3;H0;!$(NC=O)]'],
    'amide': ['C(=O)N'],
    'carboxylic_acid': ['C(=O)[OX2H1]'],
    'ester': ['[#6]OC(=O)[#6]', '[#6]C(=O)O[#6]'],
    'ketone': ['[#6]C(=O)[#6]'],
    'aldehyde': ['[CX3H1]=O'],
    'alcohol': ['[OX2H]'],
    'ether': ['[#6]O[#6]'],
    'phenyl': ['c1ccccc1'],
    'phenol': ['Oc1ccccc1'],
    'aniline': ['Nc1ccccc1'],
    'nitro': ['[$([N+](=O)[O-])]'],
    'nitrile': ['[#6]#[#7]'],
    'sulfonyl': ['S(=O)(=O)'],
    'sulfide': ['[#6][SX2][#6]'],
    'thiol': ['[SX2H]'],
    'fluorine': ['[F]'],
    'chlorine': ['[Cl]'],
    'bromine': ['[Br]'],
    'iodine': ['[I]'],
    'methyl': ['[CX4;H3]'],
    'ethyl': ['[CX4;H2][CX4;H3]'],
    'propyl': ['[CX4;H2][CX4;H2][CX4;H3]'],
    'isopropyl': ['[CX4H]([CX4H3])[CX4H3]'],
    'butyl': ['[CX4;H2][CX4;H2][CX4;H2][CX4H3]'],
    'tert_butyl': ['[CX4]([CX4H3])([CX4H3])[CX4H3]'],
    'vinyl': ['[CX3]=[CX3]'],
    'acetyl': ['[CH3]C(=O)'],
    'benzyl': ['[CX4]c1ccccc1'],
    'methoxy': ['[#6][OX2][CH3]'],
    'ethoxy': ['[#6][OX2][CH2][CH3]'],
    'hydroxyl': ['[OX2H]'],
    'carbonyl': ['[CX3]=[OX1]'],
    'cyano': ['[#6]#[#7]'],
    'formyl': ['[CH]=O'],
    'imine': ['[CX3]=[NX2]'],
    'azo': ['[NX2]=[NX2]'],
    'thiourea': ['NC(=S)N'],
    'urea': ['NC(=O)N'],
    'guanidine': ['N=C(N)N'],
    'pyridine': ['c1ccncc1'],
    'thiophene': ['c1ccsc1'],
    'furan': ['c1ccoc1'],
    'imidazole': ['c1c[nH]cn1'],
    'pyrrole': ['c1c[nH]cc1'],
    'thiazole': ['c1nccs1', 'n1ccsc1'],
    'pyrazole': ['c1cn[nH]c1'],
    'piperidine': ['C1CCNCC1'],
    'morpholine': ['C1COCCN1'],
}

FG_PATTERNS = FG_SMARTS


def _compile_fg_queries(fg_smarts):
    queries = {}
    bad = []
    for name, lst in fg_smarts.items():
        qlist = []
        for smarts in lst:
            q = Chem.MolFromSmarts(smarts)
            if q is not None:
                qlist.append(q)
            else:
                bad.append((name, smarts))
        queries[name] = qlist
    if bad:
        print(f"SMARTS 解析失败（已跳过）: {bad[:10]} … 共 {len(bad)} 条")
    return queries


FG_QUERY_MOLS = _compile_fg_queries(FG_SMARTS)


def _mol_from_smiles(smi):
    s = str(smi).strip()
    mol = Chem.MolFromSmiles(s)
    if mol is not None:
        return mol
    mol = Chem.MolFromSmiles(s, sanitize=False)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return None
    return mol


def count_fg_in_smiles(smiles_str, query_map=None):
    """RDKit：SMARTS 子结构匹配，各类 FG 是否出现（0/1）。"""
    if query_map is None:
        query_map = FG_QUERY_MOLS
    if pd.isna(smiles_str) or str(smiles_str) in ('', 'nan', 'None'):
        return {k: 0 for k in query_map}
    mol = _mol_from_smiles(smiles_str)
    if mol is None:
        return {k: 0 for k in query_map}
    out = {}
    for fg_name, qmols in query_map.items():
        hit = 0
        for q in qmols:
            if mol.HasSubstructMatch(q):
                hit = 1
                break
        out[fg_name] = hit
    return out


# Test on first few SMILES
test_smiles = high_pce_real['A_smiles_1'].dropna().iloc[0]
print(f"Test SMILES: {test_smiles}")
test_fg = count_fg_in_smiles(test_smiles)
print(f"FG found (RDKit): {[k for k, v in test_fg.items() if v > 0]}")

# Extract FG from all real high PCE samples
print("Extracting functional groups from real high PCE samples...")
real_fg_counts = {fg: 0 for fg in FG_PATTERNS.keys()}
real_total = 0

for idx, row in high_pce_real.iterrows():
    sample_fgs = set()
    for smi_col in available_smiles:
        smiles = row.get(smi_col, '')
        if pd.isna(smiles) or str(smiles) in ['nan', '', 'None']:
            continue
        fg_dict = count_fg_in_smiles(str(smiles))
        for fg, count in fg_dict.items():
            if count > 0:
                sample_fgs.add(fg)

    for fg in sample_fgs:
        real_fg_counts[fg] += 1
    real_total += 1

# Compute frequencies
real_fg_freq = {fg: count / real_total for fg, count in real_fg_counts.items()}

print(f"\nReal high PCE samples analyzed: {real_total}")
print(f"Top 10 functional groups in real high PCE samples:")
sorted_real_fg = sorted(real_fg_freq.items(), key=lambda x: x[1], reverse=True)
for fg, freq in sorted_real_fg[:10]:
    print(f"  {fg}: {freq:.4f} ({real_fg_counts[fg]}/{real_total})")

# Now extract FG from all samples for baseline comparison
print(f"\nExtracting FG from all valid samples for baseline...")
all_fg_counts = {fg: 0 for fg in FG_PATTERNS.keys()}
all_total = 0

for idx, row in df_valid_reset.iterrows():
    sample_fgs = set()
    for smi_col in available_smiles:
        smiles = row.get(smi_col, '')
        if pd.isna(smiles) or str(smiles) in ['nan', '', 'None']:
            continue
        fg_dict = count_fg_in_smiles(str(smiles))
        for fg, count in fg_dict.items():
            if count > 0:
                sample_fgs.add(fg)

    for fg in sample_fgs:
        all_fg_counts[fg] += 1
    all_total += 1

all_fg_freq = {fg: count / all_total for fg, count in all_fg_counts.items()}

# Compute enrichment ratio
enrichment = {}
for fg in FG_PATTERNS.keys():
    if all_fg_freq[fg] > 0:
        enrichment[fg] = real_fg_freq[fg] / all_fg_freq[fg]
    else:
        enrichment[fg] = float('inf') if real_fg_freq[fg] > 0 else 0

print(f"\nTop 10 enriched functional groups (high PCE vs all):")
sorted_enrichment = sorted(enrichment.items(), key=lambda x: x[1] if x[1] != float('inf') else 999, reverse=True)
for fg, enr in sorted_enrichment[:10]:
    print(f"  {fg}: enrichment={enr:.2f}x (high={real_fg_freq[fg]:.4f}, all={all_fg_freq[fg]:.4f})")


Test SMILES: C(=[NH2+])N
FG found (RDKit): ['amine_primary']
Extracting functional groups from real high PCE samples...

Real high PCE samples analyzed: 1948
Top 10 functional groups in real high PCE samples:
  iodine: 0.9168 (1786/1948)
  amine_primary: 0.8932 (1740/1948)
  methyl: 0.5118 (997/1948)
  bromine: 0.3691 (719/1948)
  chlorine: 0.0282 (55/1948)
  phenyl: 0.0026 (5/1948)
  benzyl: 0.0026 (5/1948)
  ethyl: 0.0021 (4/1948)
  propyl: 0.0021 (4/1948)
  butyl: 0.0021 (4/1948)

Extracting FG from all valid samples for baseline...


[16:55:10] SMILES Parse Error: syntax error while parsing: [Yt+2]
[16:55:10] SMILES Parse Error: check for mistakes around position 3:
[16:55:10] [Yt+2]
[16:55:10] ~~^
[16:55:10] SMILES Parse Error: Failed parsing SMILES '[Yt+2]' for input: '[Yt+2]'
[16:55:10] SMILES Parse Error: syntax error while parsing: [Yt+2]
[16:55:10] SMILES Parse Error: check for mistakes around position 3:
[16:55:10] [Yt+2]
[16:55:10] ~~^
[16:55:10] SMILES Parse Error: Failed parsing SMILES '[Yt+2]' for input: '[Yt+2]'
[16:55:10] SMILES Parse Error: syntax error while parsing: [Yt+2]
[16:55:10] SMILES Parse Error: check for mistakes around position 3:
[16:55:10] [Yt+2]
[16:55:10] ~~^
[16:55:10] SMILES Parse Error: Failed parsing SMILES '[Yt+2]' for input: '[Yt+2]'
[16:55:10] SMILES Parse Error: syntax error while parsing: [Yt+2]
[16:55:10] SMILES Parse Error: check for mistakes around position 3:
[16:55:10] [Yt+2]
[16:55:10] ~~^
[16:55:10] SMILES Parse Error: Failed parsing SMILES '[Yt+2]' for input: '[Yt+2]'



Top 10 enriched functional groups (high PCE vs all):
  pyridine: enrichment=1.86x (high=0.0010, all=0.0006)
  amine_primary: enrichment=1.42x (high=0.8932, all=0.6309)
  imidazole: enrichment=1.24x (high=0.0015, all=0.0012)
  iodine: enrichment=1.03x (high=0.9168, all=0.8893)
  methyl: enrichment=0.90x (high=0.5118, all=0.5699)
  bromine: enrichment=0.89x (high=0.3691, all=0.4165)
  chlorine: enrichment=0.85x (high=0.0282, all=0.0330)
  fluorine: enrichment=0.75x (high=0.0010, all=0.0014)
  carboxylic_acid: enrichment=0.44x (high=0.0010, all=0.0023)
  alcohol: enrichment=0.44x (high=0.0010, all=0.0023)


In [114]:

# For virtual recipes, we need to map feature values back to SMILES
# Strategy: For each virtual recipe, find the nearest neighbor in real data
# and use that neighbor's SMILES for FG analysis

from sklearn.neighbors import NearestNeighbors

# Get the valid feature matrix (7266 samples)
X_valid_features = feature_df[valid_mask_full].values

# Find nearest neighbors for virtual recipes
nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
nn.fit(X_valid_features)

# For each virtual recipe, find nearest real recipe
distances, indices = nn.kneighbors(virtual_orig_unscaled)

print(f"Virtual to real NN distances: mean={distances.mean():.2f}, max={distances.max():.2f}")

# Extract FG from nearest real samples
virtual_fg_counts = {fg: 0 for fg in FG_PATTERNS.keys()}
virtual_total = 0

for i, nn_idx in enumerate(indices.flatten()):
    real_row = df_valid_reset.iloc[nn_idx]
    sample_fgs = set()
    
    for smi_col in available_smiles:
        smiles = real_row.get(smi_col, '')
        if pd.isna(smiles) or str(smiles) in ['nan', '', 'None']:
            continue
        fg_dict = count_fg_in_smiles(str(smiles))
        for fg, count in fg_dict.items():
            if count > 0:
                sample_fgs.add(fg)
    
    for fg in sample_fgs:
        virtual_fg_counts[fg] += 1
    virtual_total += 1

# Compute frequencies
virtual_fg_freq = {fg: count / virtual_total for fg, count in virtual_fg_counts.items()}

print(f"\nVirtual recipes analyzed: {virtual_total}")
print(f"Top 10 functional groups in virtual recipes:")
sorted_virtual_fg = sorted(virtual_fg_freq.items(), key=lambda x: x[1], reverse=True)
for fg, freq in sorted_virtual_fg[:10]:
    print(f"  {fg}: {freq:.4f} ({virtual_fg_counts[fg]}/{virtual_total})")

# Save real FG commonality
real_fg_df = pd.DataFrame([
    {
        'functional_group': fg,
        'high_pce_count': real_fg_counts[fg],
        'high_pce_frequency': real_fg_freq[fg],
        'all_count': all_fg_counts[fg],
        'all_frequency': all_fg_freq[fg],
        'enrichment_ratio': enrichment[fg] if enrichment[fg] != float('inf') else 999
    }
    for fg in FG_PATTERNS.keys()
])
real_fg_df = real_fg_df.sort_values('high_pce_frequency', ascending=False)
real_fg_df.to_csv('output/real_fg_commonality.csv', index=False)
print(f"\nSaved: output/real_fg_commonality.csv")

# Save virtual FG commonality
virtual_fg_df = pd.DataFrame([
    {
        'functional_group': fg,
        'virtual_count': virtual_fg_counts[fg],
        'virtual_frequency': virtual_fg_freq[fg]
    }
    for fg in FG_PATTERNS.keys()
])
virtual_fg_df = virtual_fg_df.sort_values('virtual_frequency', ascending=False)
virtual_fg_df.to_csv('output/virtual_fg_commonality.csv', index=False)
print(f"Saved: output/virtual_fg_commonality.csv")


Virtual to real NN distances: mean=853.31, max=4401.73

Virtual recipes analyzed: 100
Top 10 functional groups in virtual recipes:
  iodine: 0.9500 (95/100)
  methyl: 0.6500 (65/100)
  amine_primary: 0.5300 (53/100)
  bromine: 0.3400 (34/100)
  chlorine: 0.1000 (10/100)
  phenyl: 0.0100 (1/100)
  ethyl: 0.0100 (1/100)
  benzyl: 0.0100 (1/100)
  amine_secondary: 0.0000 (0/100)
  amine_tertiary: 0.0000 (0/100)

Saved: output/real_fg_commonality.csv
Saved: output/virtual_fg_commonality.csv


In [115]:

# Compare real vs virtual functional group commonality

# Get top 10 FG from each
top_n = 10
real_top10 = set([fg for fg, _ in sorted_real_fg[:top_n]])
virtual_top10 = set([fg for fg, _ in sorted_virtual_fg[:top_n]])

print(f"Real Top-10 FG: {sorted(real_top10)}")
print(f"Virtual Top-10 FG: {sorted(virtual_top10)}")

# Compute overlap
overlap = real_top10 & virtual_top10
consistency_score = len(overlap) / top_n
print(f"\nOverlap: {sorted(overlap)}")
print(f"Consistency score (overlap): {consistency_score:.2f} ({len(overlap)}/{top_n})")

# 全 51 维频率向量上的 Pearson 易被「零-零对」抬高；报告至少一侧非零子集的相关作对照
fg_order = list(FG_PATTERNS.keys())
r_arr = np.array([real_fg_freq[f] for f in fg_order])
v_arr = np.array([virtual_fg_freq[f] for f in fg_order])
nz_mask = (r_arr > 0) | (v_arr > 0)
if nz_mask.sum() >= 2:
    pr_nz = np.corrcoef(r_arr[nz_mask], v_arr[nz_mask])[0, 1]
    print(f"\nPearson r (freq, non-zero subset): {pr_nz:.4f} (n={nz_mask.sum()}/{len(fg_order)} FG)")
print("(解读：优先参考 Top-10 重叠；全向量 r 易虚高)")

# Create comparison DataFrame
comparison_data = []
for fg in FG_PATTERNS.keys():
    comparison_data.append({
        'functional_group': fg,
        'real_high_pce_freq': real_fg_freq.get(fg, 0),
        'virtual_freq': virtual_fg_freq.get(fg, 0),
        'all_freq': all_fg_freq.get(fg, 0),
        'enrichment_ratio': enrichment.get(fg, 0) if enrichment.get(fg, 0) != float('inf') else 999,
        'in_real_top10': fg in real_top10,
        'in_virtual_top10': fg in virtual_top10
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df['freq_difference'] = comparison_df['virtual_freq'] - comparison_df['real_high_pce_freq']
comparison_df = comparison_df.sort_values('real_high_pce_freq', ascending=False)
comparison_df.to_csv('output/fg_comparison_real_vs_virtual.csv', index=False)
print(f"\nSaved: output/fg_comparison_real_vs_virtual.csv")

# Use virtual commonality to guide real library screening
# Extract virtual common features pattern
virtual_common_fgs = set([fg for fg, freq in sorted_virtual_fg[:5] if freq > 0.5])
print(f"\nVirtual common FGs (freq > 0.5): {sorted(virtual_common_fgs)}")

# Screen real library for samples matching virtual commonality pattern
screened_indices = []
for idx, row in df_valid_reset.iterrows():
    sample_fgs = set()
    for smi_col in available_smiles:
        smiles = row.get(smi_col, '')
        if pd.isna(smiles) or str(smiles) in ['nan', '', 'None']:
            continue
        fg_dict = count_fg_in_smiles(str(smiles))
        for fg, count in fg_dict.items():
            if count > 0:
                sample_fgs.add(fg)
    
    # Check if sample has at least 3 of the virtual common FGs
    overlap_count = len(sample_fgs & virtual_common_fgs)
    if overlap_count >= 3:
        screened_indices.append(idx)

screened_df = df_valid_reset.iloc[screened_indices].copy()
print(f"\nScreened recipes (match >= 3 virtual common FGs): {len(screened_df)}")
if len(screened_df) > 0:
    print(f"Screened PCE: mean={screened_df['PCE'].mean():.2f}%, median={screened_df['PCE'].median():.2f}%")
    print(f"Screened PCE >= 22%: {(screened_df['PCE'] >= 22).sum()}/{len(screened_df)}")
    print(f"All data PCE: mean={df_valid_reset['PCE'].mean():.2f}%, median={df_valid_reset['PCE'].median():.2f}%")


Real Top-10 FG: ['amine_primary', 'benzyl', 'bromine', 'butyl', 'chlorine', 'ethyl', 'iodine', 'methyl', 'phenyl', 'propyl']
Virtual Top-10 FG: ['amine_primary', 'amine_secondary', 'amine_tertiary', 'benzyl', 'bromine', 'chlorine', 'ethyl', 'iodine', 'methyl', 'phenyl']

Overlap: ['amine_primary', 'benzyl', 'bromine', 'chlorine', 'ethyl', 'iodine', 'methyl', 'phenyl']
Consistency score (overlap): 0.80 (8/10)

Pearson r (freq, non-zero subset): 0.9502 (n=17/51 FG)
(解读：优先参考 Top-10 重叠；全向量 r 易虚高)

Saved: output/fg_comparison_real_vs_virtual.csv

Virtual common FGs (freq > 0.5): ['amine_primary', 'iodine', 'methyl']


[16:55:12] SMILES Parse Error: syntax error while parsing: [Yt+2]
[16:55:12] SMILES Parse Error: check for mistakes around position 3:
[16:55:12] [Yt+2]
[16:55:12] ~~^
[16:55:12] SMILES Parse Error: Failed parsing SMILES '[Yt+2]' for input: '[Yt+2]'
[16:55:12] SMILES Parse Error: syntax error while parsing: [Yt+2]
[16:55:12] SMILES Parse Error: check for mistakes around position 3:
[16:55:12] [Yt+2]
[16:55:12] ~~^
[16:55:12] SMILES Parse Error: Failed parsing SMILES '[Yt+2]' for input: '[Yt+2]'
[16:55:13] SMILES Parse Error: syntax error while parsing: [Yt+2]
[16:55:13] SMILES Parse Error: check for mistakes around position 3:
[16:55:13] [Yt+2]
[16:55:13] ~~^
[16:55:13] SMILES Parse Error: Failed parsing SMILES '[Yt+2]' for input: '[Yt+2]'
[16:55:13] SMILES Parse Error: syntax error while parsing: [Yt+2]
[16:55:13] SMILES Parse Error: check for mistakes around position 3:
[16:55:13] [Yt+2]
[16:55:13] ~~^
[16:55:13] SMILES Parse Error: Failed parsing SMILES '[Yt+2]' for input: '[Yt+2]'



Screened recipes (match >= 3 virtual common FGs): 2681
Screened PCE: mean=19.91%, median=20.80%
Screened PCE >= 22%: 958/2681
All data PCE: mean=17.90%, median=19.32%
